# Diamond Dataset — Interactive Visualization Assignment
**Dataset:** `seaborn` diamonds | **Library:** Plotly

**Description:** This notebook contained an analysis of how alter perceptions of different statistics will affect the distribution presented in the graphs through implementations of interactive features.

**The statistics that will be evaluated are:**
- Skewness
- Nonlinearity
- Density
- Range
- Resolution

In [1]:
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
from ipywidgets import interact, widgets
import warnings
warnings.filterwarnings('ignore')

df = sns.load_dataset('diamonds')
print(df.shape)
df.head()

(53940, 10)


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


---
## Part 1 — Baseline Interactive View

In [ ]:
fig1 = px.scatter(
    df,
    x='carat',
    y='price',
    color='cut',
    title='Carat vs Price — Colored by Cut',
    labels={'carat': 'Carat Weight', 'price': 'Price (USD)'},
    opacity=0.4,
    hover_data=['cut', 'color', 'clarity', 'depth', 'table'],
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig1.update_layout(
    height=550,
    legend_title='Cut Quality',
    plot_bgcolor='#f8f9fa'
)
fig1.show()

### Part 1 Interpretation

**Does the relationship between carat and price appear linear?**  
No. While there is a clear positive trend — higher carat weight generally corresponds to higher price — the relationship is curved. Price accelerates as carat increases, suggesting an exponential or power-law relationship rather than a straight line. The spread (variance) of price also widens dramatically at higher carats, which is inconsistent with a linear model.

**Where does overplotting obscure density?**  
The most severe overplotting occurs in the 0.2–1.0 carat range, especially at price levels below \$5,000. At these values, tens of thousands of points stack on top of each other, making it impossible to tell whether there is a dense cluster or simply a few hundred points. The lower-left quadrant appears as a near-solid mass.

**What does hover reveal that the visual field does not?**  
Hover reveals the specific cut quality, color grade, and clarity of individual diamonds. Visually, it is impossible to distinguish, say, an Ideal-cut VS1 diamond from a Fair-cut SI2 diamond when they share the same (carat, price) coordinate. Hover also reveals exact price values that are invisible in overlapping clusters.

---
## Part 2 — Interactive Scale Toggle

In [ ]:
# Build figure with two traces per cut — one for linear, one for log
cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
colors = px.colors.qualitative.Bold

fig2 = go.Figure()

for i, cut in enumerate(cut_order):
    sub = df[df['cut'] == cut]
    # Linear trace (visible by default)
    fig2.add_trace(go.Scatter(
        x=sub['carat'], y=sub['price'],
        mode='markers',
        marker=dict(color=colors[i], opacity=0.35, size=3),
        name=cut,
        legendgroup=cut,
        visible=True,
        customdata=sub[['cut','color','clarity']].values,
        hovertemplate='<b>%{customdata[0]}</b><br>Carat: %{x:.2f}<br>Price: $%{y:,.0f}<br>Color: %{customdata[1]} | Clarity: %{customdata[2]}<extra></extra>'
    ))

fig2.update_layout(
    title='Carat vs Price — Linear vs Log Y-Axis',
    xaxis_title='Carat Weight',
    yaxis_title='Price (USD)',
    height=550,
    plot_bgcolor='#f8f9fa',
    updatemenus=[
        dict(
            type='buttons',
            direction='right',
            x=0.5, y=1.12,
            xanchor='center',
            showactive=True,
            buttons=[
                dict(
                    label='Linear Y-Axis',
                    method='relayout',
                    args=[{'yaxis.type': 'linear', 'yaxis.title.text': 'Price (USD)'}]
                ),
                dict(
                    label='Log Y-Axis',
                    method='relayout',
                    args=[{'yaxis.type': 'log', 'yaxis.title.text': 'Price — Log Scale (USD)'}]
                )
            ]
        )
    ]
)
fig2.show()

### Part 2 Interpretation

**How does log scaling change perceived slope?**  
On a log y-axis, the steep upward curve in the linear view flattens into something that looks closer to a straight diagonal. This is because log compression reduces the visual distance between large values. A jump from \$5,000 to \$10,000 looks the same as a jump from \$500 to \$1,000, even though the absolute difference is 10× larger in the first case.

**Does the relationship appear more linear?**  
Yes — more so. Under log scale, carat and log(price) follow a visually tighter linear path. This suggests the true relationship may be closer to `log(price) ∝ carat`, or equivalently a power/exponential relationship on the original scale.

**What structural feature becomes visible under log scale?**  
Under log scale, the fan-shaped variance expansion at high carats collapses, and distinct *horizontal bands* become visible — plateaus where many diamonds cluster at similar log-prices despite varying carat weights. These likely correspond to psychological pricing thresholds (e.g., \$1,000, \$2,000, \$5,000) that are invisible on the linear scale.

**When would log scaling be misleading?**  
Log scaling is misleading when the audience interprets visual distance as absolute difference rather than proportional difference. A viewer might conclude that a 1-carat diamond costs only "a little more" than a 0.5-carat diamond, when in fact the absolute price gap may be several thousand dollars.

---
## Part 3 — Interactive Range vs Filtering
### 3A — Zoom (axis restriction via range selector)

In [ ]:
fig3a = px.scatter(
    df,
    x='carat', y='price',
    color='cut',
    title='Part 3A — Zoom: All data retained, axis range restricted',
    opacity=0.35,
    labels={'carat': 'Carat', 'price': 'Price (USD)'},
    color_discrete_sequence=px.colors.qualitative.Bold
)

fig3a.update_layout(
    height=520,
    plot_bgcolor='#f8f9fa',
    xaxis=dict(
        rangeslider=dict(visible=True),
        range=[0, 1.5]
    ),
    yaxis=dict(range=[0, 5000])
)
fig3a.show()

### 3B — Data Filtering via Price Slider

In [ ]:
price_steps = [1000, 2500, 5000, 10000, 18823]
price_labels = ['≤$1K', '≤$2.5K', '≤$5K', '≤$10K', 'All']

fig3b = go.Figure()
colors = px.colors.qualitative.Bold
cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']

# For each price ceiling, add a set of traces (one per cut)
for p_idx, p_max in enumerate(price_steps):
    sub = df[df['price'] <= p_max]
    for c_idx, cut in enumerate(cut_order):
        csub = sub[sub['cut'] == cut]
        fig3b.add_trace(go.Scatter(
            x=csub['carat'], y=csub['price'],
            mode='markers',
            marker=dict(color=colors[c_idx], opacity=0.35, size=3),
            name=cut,
            legendgroup=cut,
            showlegend=(p_idx == 4),  # only show legend for last step
            visible=(p_idx == 4)  # start with all data shown
        ))

# Build slider steps — each step shows 5 traces (one per cut)
steps = []
for p_idx, label in enumerate(price_labels):
    visibility = [False] * (len(price_steps) * len(cut_order))
    for c_idx in range(len(cut_order)):
        visibility[p_idx * len(cut_order) + c_idx] = True
    steps.append(dict(
        method='update',
        label=label,
        args=[{'visible': visibility},
              {'title': f'Part 3B — Filtered: Price {label}'}]
    ))

fig3b.update_layout(
    title='Part 3B — Data Filtering by Price Range',
    xaxis_title='Carat', yaxis_title='Price (USD)',
    height=550,
    plot_bgcolor='#f8f9fa',
    sliders=[dict(
        active=4,
        currentvalue={'prefix': 'Max Price: ', 'font': {'size': 13}},
        pad={'t': 50},
        steps=steps
    )]
)
fig3b.show()

### Part 3 Interpretation

**What remains visible when zooming?**  
All data points still exist in the dataset — the zoom simply restricts the visible window of the axes. Points outside the viewport are not gone; they can be recovered by panning or adjusting the range. The overall shape and slope of the relationship within that window reflects the real distribution at those values.

**What disappears when filtering?**  
When filtering by price ceiling, all diamonds above that threshold are removed from the dataset fed to the chart. High-carat, high-price diamonds vanish entirely. This shrinks the visible x-range and removes the part of the relationship where price growth accelerates most steeply.

**How can filtering change perceived slope?**  
Filtering to only low-priced diamonds cuts off the upper-right portion of the scatter — the region where carat weight rises sharply with price. What remains looks nearly flat or only mildly positive, making it appear that carat has little influence on price. This is a form of *range restriction bias*: artificially constraining the range of one variable suppresses the apparent correlation.

**Which approach carries greater risk of misleading interpretation?**  
Filtering carries greater risk. With zooming, a careful viewer can see the axis labels and understand they are looking at a subset of the range. With filtering, the data itself is silently altered — a viewer has no visual cue that points have been removed, and may incorrectly treat the filtered view as a complete picture.

---
## Part 4 — Interactive Bin Resolution

In [ ]:
bin_levels = [10, 20, 40, 70, 100]
fig4 = go.Figure()

for b_idx, nbins in enumerate(bin_levels):
    fig4.add_trace(go.Histogram(
        x=df['price'],
        nbinsx=nbins,
        name=f'{nbins} bins',
        marker_color='steelblue',
        visible=(b_idx == 1)  # start at 20 bins
    ))

steps = []
for b_idx, nbins in enumerate(bin_levels):
    visibility = [False] * len(bin_levels)
    visibility[b_idx] = True
    steps.append(dict(
        method='update',
        label=str(nbins),
        args=[{'visible': visibility},
              {'title': f'Price Distribution — {nbins} Bins'}]
    ))

fig4.update_layout(
    title='Price Distribution — 20 Bins',
    xaxis_title='Price (USD)',
    yaxis_title='Count',
    height=500,
    plot_bgcolor='#f8f9fa',
    sliders=[dict(
        active=1,
        currentvalue={'prefix': 'Bin Count: ', 'font': {'size': 13}},
        pad={'t': 50},
        steps=steps
    )]
)
fig4.show()

### Part 4 Interpretation

**What does low binning hide?**  
With only 10 bins, the distribution appears as a smooth, roughly right-skewed hill. This conceals internal structure: sub-peaks, plateaus, and gaps that exist within the price range. The distinct spike at very low prices (\$300–600) is absorbed into the first bin and disappears. The secondary cluster around \$1,000–1,200 is also masked.

**What does high binning exaggerate?**  
With 100 bins, every minor fluctuation in count becomes visually salient. Gaps between bins that are likely just sampling noise start to look like meaningful structural breaks in the distribution. This creates a false sense of complexity — the viewer may read "bimodal" or "multimodal" structure into what is actually sampling variation.

**Which bin level would you use in a presentation?**  
Around 40 bins. This level clearly shows the right skew, the concentration of diamonds at lower prices, and the long tail toward high-price diamonds, without introducing misleading noise.

**Why is binning an interpretive decision rather than a neutral one?**  
There is no objectively "correct" bin count. The analyst's choice of binning encodes an implicit claim about what level of detail is meaningful. A fine-grained histogram implies the micro-fluctuations are real signal; a coarse one implies they are not. Both are arguments about the data, made through visual design.

---
## Part 5 — Overplotting & Transparency

In [ ]:
alpha_levels = [1.0, 0.6, 0.3, 0.15, 0.05]
alpha_labels = ['Opaque (1.0)', 'Moderate (0.6)', 'Medium (0.3)', 'High (0.15)', 'Max Transparency (0.05)']

fig5 = go.Figure()

for a_idx, alpha in enumerate(alpha_levels):
    fig5.add_trace(go.Scatter(
        x=df['carat'], y=df['price'],
        mode='markers',
        marker=dict(
            color='steelblue',
            opacity=alpha,
            size=3
        ),
        name=f'alpha={alpha}',
        visible=(a_idx == 2)  # start at 0.3
    ))

steps = []
for a_idx, label in enumerate(alpha_labels):
    visibility = [False] * len(alpha_levels)
    visibility[a_idx] = True
    steps.append(dict(
        method='update',
        label=label,
        args=[{'visible': visibility},
              {'title': f'Carat vs Price — {label}'}]
    ))

fig5.update_layout(
    title='Carat vs Price — Adjust Transparency',
    xaxis_title='Carat',
    yaxis_title='Price (USD)',
    height=520,
    plot_bgcolor='#f8f9fa',
    sliders=[dict(
        active=2,
        currentvalue={'prefix': 'Transparency: ', 'font': {'size': 13}},
        pad={'t': 50},
        steps=steps
    )]
)
fig5.show()

### Part 5 Interpretation

**Where does overplotting distort density perception?**  
The worst overplotting occurs at low carat values (0.2–1.0) and low prices (\$300–\$3,000). At full opacity, this region appears as a uniform dark mass, making it impossible to distinguish whether it contains 100 points or 10,000. Specific clustering patterns — for example, higher concentrations at round carat values like 0.5, 0.7, and 1.0 — are entirely hidden.

**How does transparency change interpretation?**  
As transparency increases, the overlapping region begins to lighten, and areas of genuinely high density remain darker than sparse areas. Viewers can now see that density is highest in a narrow band below 1 carat. Transparency converts opacity into a rough density signal, revealing the actual shape of the distribution rather than just its footprint.

**What limitation remains even after adjusting transparency?**  
Transparency is not a calibrated density measure. Even at high transparency settings, a region with 500 overlapping points cannot be reliably distinguished from one with 5,000 overlapping points. Additionally, screen rendering and rendering in different environments affects how transparency looks, so the same alpha value may communicate different densities depending on the viewer's display. True density estimation requires a 2D density plot (e.g., hex-bin or KDE), not opacity tuning.

---
## Part 6 — Final Interactive Dashboard

In [ ]:
from plotly.subplots import make_subplots

cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
colors = px.colors.qualitative.Bold
bin_options = [10, 25, 50, 100]
price_ceilings = [2000, 5000, 10000, 18823]
price_labels = ['≤$2K', '≤$5K', '≤$10K', 'All']

fig6 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Carat vs Price by Cut', 'Price Distribution'),
    column_widths=[0.6, 0.4]
)

# We'll pre-build all combinations: (price_ceiling × bins)
# Each combination = 5 scatter traces (by cut) + 1 histogram trace
# Total trace count = len(price_ceilings) * len(bin_options) * 6

for p_idx, p_max in enumerate(price_ceilings):
    sub = df[df['price'] <= p_max]
    for b_idx, nbins in enumerate(bin_options):
        combo_idx = p_idx * len(bin_options) + b_idx
        is_visible = (p_idx == 3 and b_idx == 1)  # default: All + 25 bins

        # Scatter traces
        for c_idx, cut in enumerate(cut_order):
            csub = sub[sub['cut'] == cut]
            fig6.add_trace(go.Scatter(
                x=csub['carat'], y=csub['price'],
                mode='markers',
                marker=dict(color=colors[c_idx], opacity=0.3, size=2.5),
                name=cut,
                legendgroup=cut,
                showlegend=(combo_idx == (3 * len(bin_options) + 1) and c_idx == c_idx),
                visible=is_visible,
                hovertemplate=f'Cut: {cut}<br>Carat: %{{x:.2f}}<br>Price: $%{{y:,.0f}}<extra></extra>'
            ), row=1, col=1)

        # Histogram trace
        fig6.add_trace(go.Histogram(
            x=sub['price'],
            nbinsx=nbins,
            marker_color='steelblue',
            opacity=0.75,
            name=f'Histogram ({nbins} bins)',
            showlegend=False,
            visible=is_visible,
            hovertemplate='Price: $%{x:,.0f}<br>Count: %{y}<extra></extra>'
        ), row=1, col=2)

traces_per_combo = len(cut_order) + 1  # 5 scatter + 1 hist
total_combos = len(price_ceilings) * len(bin_options)

# Build price filter dropdown buttons
price_buttons = []
for p_idx, p_label in enumerate(price_labels):
    visibility = [False] * (total_combos * traces_per_combo)
    # Show all bin combos for this price (so bin slider still works)
    # But we default to showing only the currently selected bin
    # Strategy: dropdown sets price; bin slider uses separate mechanism
    # Simple approach: each button activates ALL bin levels for that price — 
    # then user uses bin slider to select bin. But Plotly doesn't support
    # stacked state easily, so we show only (price × default bin=1)
    default_b = 1  # index for 25 bins
    combo_idx = p_idx * len(bin_options) + default_b
    base = combo_idx * traces_per_combo
    for t in range(traces_per_combo):
        visibility[base + t] = True
    price_buttons.append(dict(
        label=p_label,
        method='update',
        args=[{'visible': visibility},
              {'title.text': f'Diamond Dashboard — Price Filter: {p_label} | 25 Bins'}]
    ))

# Build bin slider steps (use default price=All, index=3)
bin_steps = []
default_p = 3  # All
for b_idx, nbins in enumerate(bin_options):
    visibility = [False] * (total_combos * traces_per_combo)
    combo_idx = default_p * len(bin_options) + b_idx
    base = combo_idx * traces_per_combo
    for t in range(traces_per_combo):
        visibility[base + t] = True
    bin_steps.append(dict(
        method='update',
        label=str(nbins),
        args=[{'visible': visibility},
              {'title.text': f'Diamond Dashboard — Price Filter: All | {nbins} Bins'}]
    ))

fig6.update_layout(
    title='Diamond Dashboard — Price Filter: All | 25 Bins',
    height=580,
    plot_bgcolor='#f8f9fa',
    paper_bgcolor='white',
    updatemenus=[
        dict(
            type='dropdown',
            direction='down',
            x=0.0, y=1.15,
            xanchor='left',
            showactive=True,
            buttons=price_buttons,
            bgcolor='white',
            bordercolor='gray'
        )
    ],
    sliders=[
        dict(
            active=1,
            currentvalue={'prefix': 'Histogram Bins: ', 'font': {'size': 12}},
            pad={'t': 55, 'b': 10},
            steps=bin_steps,
            x=0.0,
            len=0.5
        )
    ]
)

fig6.update_xaxes(title_text='Carat Weight', row=1, col=1)
fig6.update_yaxes(title_text='Price (USD)', row=1, col=1)
fig6.update_xaxes(title_text='Price (USD)', row=1, col=2)
fig6.update_yaxes(title_text='Count', row=1, col=2)

fig6.show()

---
## Part 7 — Final Conceptual Reflection

### 1. How did your understanding of the carat–price relationship change?

The baseline linear scatter suggested a positive, roughly monotonic relationship between carat and price. It implied that adding weight to a diamond increases its value in a proportional, consistent way. This is a seductive simplification: the cloud of points rises left to right, and the eye constructs a mental regression line.

Interaction dismantled that story. Log scaling revealed that the relationship is better described as multiplicative rather than additive — price does not increase by a fixed dollar amount per carat, it increases by a fixed *proportion*. Filtering exposed range restriction bias: when only low-priced diamonds are shown, the slope almost vanishes, suggesting that the apparent strength of the carat–price relationship depends on including the high end of the range. Transparency and binning revealed that the "cloud" in the baseline view is actually a structured density distribution — highest around 0.3–1.0 carat, thinning into a long tail.

The most important structural feature that only became visible through interaction was the *clustering at round carat values* (0.5, 0.7, 1.0, 1.5) — a market convention invisible in the opaque baseline but discernible under transparency and on log scale. These clusters reflect how diamonds are cut to hit psychological thresholds, not just physical weight.

---

### 2. What role did scale play in shaping perceived structure?

Linear scale renders absolute differences as equal visual distances. Because price has extreme positive skew — most diamonds cluster below \$5,000 while a small number exceed \$15,000 — the linear axis compresses the high-density region into a small portion of the plot and stretches the sparse high-price tail. This makes the relationship look explosively nonlinear and the lower price range look nearly flat.

Log scale encodes proportional differences as equal visual distances. Each doubling of price takes up the same vertical space regardless of whether it's from \$500 to \$1,000 or from \$5,000 to \$10,000. This assumption is often correct for price data, where a 10% increase has similar economic meaning at any price level. Under log scale, the carat–price relationship appears near-linear, which reveals that a power-law model (`price ≈ a × carat^b`) likely fits the data better than a linear one.

The risk of log scale is that proportional reasoning is not intuitive for general audiences. A viewer who reads visual distance as absolute difference will systematically underestimate how large the price gaps are at the high end.

---

### 3. How does resolution (binning and density) affect what counts as "pattern"?

Binning is a forced choice about granularity. At 10 bins, the price histogram shows a single right-skewed shape — a story about the overall distribution. At 100 bins, the same histogram shows a jagged, multi-peaked landscape — a story about micro-structure. Neither view is wrong, but they answer different questions and invite different conclusions.

The boundary between "pattern" and "noise" is not in the data — it is in the analyst's resolution choice. A spike in a 100-bin histogram that looks like a meaningful sub-cluster may simply be a bin that captured slightly more observations by chance. This is the epistemic danger: high resolution creates the *appearance* of precision without guaranteeing that the detail is structurally real.

In scatter plots, overplotting creates a symmetrically inverted problem: instead of too much resolution (binning noise), we have too little (opacity saturation). Dense regions look identical to moderately dense regions. Both problems are about the mismatch between visual encoding capacity and data volume.

---

### 4. What is the difference between zooming and filtering in terms of interpretive consequence?

Zooming is a *framing* operation: the dataset remains intact, but the viewer's window onto it narrows. A careful viewer can see from the axis labels that they are looking at a restricted range. The data outside the frame still exists and still shapes the relationship — it has simply moved off-screen.

Filtering is a *removal* operation: data points are deleted from the computation before the chart is drawn. The viewer has no visual evidence that any data has been removed. The resulting chart presents a partial dataset as if it were complete.

Filtering carries greater ethical risk because it can silently change the apparent strength and slope of relationships. An analyst who filters out high-priced diamonds before showing the carat–price scatter will produce a chart showing a weak or flat relationship. Without disclosure, a viewer has no way to know that the most informative part of the distribution was removed. This is the mechanism behind many cases of misleading data visualization in professional and journalistic contexts.

---

### 5. What authority does interaction give to the viewer?

Interaction distributes interpretive authority. A static chart encodes a single argument; an interactive dashboard offers a space of possible views. This is genuinely valuable when the viewer has the analytical skill to navigate that space critically — they can interrogate the data, test alternate framings, and resist a single imposed narrative.

But interaction also shifts the *burden of interpretation* onto the viewer. A static chart makes an argument the analyst stands behind. An interactive system says: *here are the controls; draw your own conclusions.* This can appear neutral while actually being evasive — the analyst still chose which controls to include, which defaults to show, and which dimensions to expose. The interaction is not objective; it is a curated set of possible views.

The risk of fragmentation is real. A viewer who settles on a filtered, low-resolution view that confirms their prior belief has not been served by the interactivity — they have been given the tools to mislead themselves. Interactivity increases understanding when the viewer is guided; it risks cherry-picking when the viewer is left entirely unguided.

---

### 6. What remains invisible in your entire interactive system?

**One assumption about the data:** The diamonds dataset assumes that each row is an independent observation. In reality, diamonds from the same mine, cutter, or retailer may be systematically similar — correlated in ways that violate independence. If the data overrepresents a particular supplier's inventory, the apparent patterns (cut distribution, price clustering) may reflect procurement decisions rather than market-wide structure.

**One structural limitation of the carat–price framing:** The dashboard treats carat and price as the primary axis of meaning. But price is also a function of color, clarity, cut, and retail context (time of purchase, vendor, market conditions). By foregrounding carat, the dashboard implicitly frames carat as *the* driver of value, invisibly downgrading the other 4Cs. A diamond with identical carat weight can vary by \$10,000 depending on clarity — a fact the dashboard structure does not highlight.

**One question the dashboard cannot answer without a different design:** The dashboard cannot show *how the carat–price relationship varies by clarity and color simultaneously*. The color-by-cut encoding occupies the one color channel available in a 2D scatter. To understand whether the price premium for a 1-carat Ideal-cut diamond differs depending on whether it is VS1 or SI2 would require either a faceted grid or a 3D visualization — neither of which is possible within the current 2-panel structure.